In [1]:
from init_env import cfg
from dataset import load_datasets, load_splits, create_ds_from_df
from models import *

import os
import json
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Flatten, GlobalAveragePooling2D
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [2]:
# Extract configuration settings
DATA_PATH = cfg["paths"]["data_path"]
MODEL_PATH = cfg["paths"]["model_path"]
IMG_SIZE = tuple(cfg["img_size"])
BATCH_SIZE = cfg["batch_size"]
USE_AUTOTUNE = cfg["use_autotune"]

DROPOUT_RATE = cfg["training"]["dropout_rate"]
FE_EPOCHS = cfg["training"]["feature_extract"]["epochs"]
FE_LR = cfg["training"]["feature_extract"]["lr"]

FT_EPOCHS = cfg["training"]["fine_tune"]["epochs"]
FT_LR = cfg["training"]["fine_tune"]["lr"]

# Load datasets
train_ds, val_ds, test_ds = load_datasets(os.path.join(DATA_PATH, "splits"), 
                                          batch_size=BATCH_SIZE,
                                          use_autotune=USE_AUTOTUNE, 
                                          img_size=IMG_SIZE)

# Check if tensorflow detects GPU
# conda install -c conda-forge cudatoolkit=11.2 cudnn=8.1 if nothing detected
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [3]:
# Stop training after 3 epochs no improvement
# Restore the best model weights
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Reduce LR when plateuing
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5, # Commonly used factor
    patience=2, # 2 epochs of no improvement
    min_lr=1e-6,
    verbose=1 # Logs when LR is reduced
)

# Model 1

In [13]:
# Import VGG16 with imagenet weights
base = VGG16(
    weights="imagenet",
    include_top=False, # Remove final layer
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

# Freeze layers
for layer in base.layers:
    layer.trainable = False

x = Flatten()(base.output)
x = Dense(512, activation="relu")(x)
# Add dropout to reduce overfitting
x = Dropout(DROPOUT_RATE)(x)
# Add final classification layer
output = Dense(3, activation="softmax")(x)
model = Model(inputs=base.input, outputs=output)

model.compile(
    optimizer=Adam(FE_LR),
    loss="sparse_categorical_crossentropy", # No need to one-hot encode
    metrics=["accuracy"]
)

model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [14]:
# Train classification layer and new layers
# Perform feature extraction
history_feature_extract = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FE_EPOCHS,
    callbacks=[early_stop, reduce_lr]
)

save_model_and_history(model, history_feature_extract, MODEL_PATH, "vgg16_feature_extract")

Epoch 1/40
1402/1402 [==============================] - 264s 178ms/step - loss: 1.0248 - accuracy: 0.7962 - val_loss: 0.7048 - val_accuracy: 0.6813 - lr: 1.0000e-04
Epoch 2/40
1402/1402 [==============================] - 254s 174ms/step - loss: 0.5383 - accuracy: 0.8248 - val_loss: 0.5981 - val_accuracy: 0.7413 - lr: 1.0000e-04
Epoch 3/40
1402/1402 [==============================] - 262s 180ms/step - loss: 0.5088 - accuracy: 0.8313 - val_loss: 0.6145 - val_accuracy: 0.7483 - lr: 1.0000e-04
Epoch 4/40
1402/1402 [==============================] - 258s 177ms/step - loss: 0.4827 - accuracy: 0.8375 - val_loss: 0.5862 - val_accuracy: 0.7798 - lr: 1.0000e-04
Epoch 5/40
1402/1402 [==============================] - 258s 176ms/step - loss: 0.4731 - accuracy: 0.8392 - val_loss: 0.5880 - val_accuracy: 0.8044 - lr: 1.0000e-04
Epoch 6/40
1402/1402 [==============================] - 261s 179ms/step - loss: 0.4628 - accuracy: 0.8449 - val_loss: 0.5475 - val_accuracy: 0.8147 - lr: 1.0000e-04
Epoch 7/40

In [15]:
unfreeze_block(model, "block5")

model.compile(
    optimizer=Adam(FT_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine_tune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FT_EPOCHS,
    callbacks=[early_stop, reduce_lr]
)

save_model_and_history(model, history_fine_tune, MODEL_PATH, "vgg16_fine_tune")

All layers in block5 unfrozen.
Epoch 1/20
1402/1402 [==============================] - 269s 182ms/step - loss: 0.3745 - accuracy: 0.8771 - val_loss: 0.4899 - val_accuracy: 0.8319 - lr: 1.0000e-05
Epoch 2/20
1402/1402 [==============================] - 262s 179ms/step - loss: 0.3479 - accuracy: 0.8836 - val_loss: 0.4903 - val_accuracy: 0.8368 - lr: 1.0000e-05
Epoch 3/20
1402/1402 [==============================] - 262s 180ms/step - loss: 0.3306 - accuracy: 0.8893 - val_loss: 0.4828 - val_accuracy: 0.8393 - lr: 1.0000e-05
Epoch 4/20
1402/1402 [==============================] - 260s 178ms/step - loss: 0.3163 - accuracy: 0.8907 - val_loss: 0.5019 - val_accuracy: 0.8408 - lr: 1.0000e-05
Epoch 5/20
1402/1402 [==============================] - ETA: 0s - loss: 0.2943 - accuracy: 0.8972     
Epoch 5: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
1402/1402 [==============================] - 259s 178ms/step - loss: 0.2943 - accuracy: 0.8972 - val_loss: 0.5014 - val_accuracy: 

# Model 2

In [8]:
# Import VGG16 with imagenet weights
base = VGG16(
    weights="imagenet",
    include_top=False, # Remove final layer
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

# Freeze layers
for layer in base.layers:
    layer.trainable = False

# Using GAP layer
x = GlobalAveragePooling2D()(base.output)
x = Dense(512, activation="relu")(x)
# Add dropout to reduce overfitting
x = Dropout(DROPOUT_RATE)(x)
# Add final classification layer
output = Dense(3, activation="softmax")(x)
model_gap = Model(inputs=base.input, outputs=output)

model_gap.compile(
    optimizer=Adam(FE_LR),
    loss="sparse_categorical_crossentropy", # No need to one-hot encode
    metrics=["accuracy"]
)

model_gap.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0   

In [9]:
# Train classification layer and new layers
# Perform feature extraction
history_gap_feature_extract = model_gap.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FE_EPOCHS,
    callbacks=[early_stop, reduce_lr]
)

save_model_and_history(model_gap, history_gap_feature_extract, MODEL_PATH, "vgg16_gap_feature_extract")

Epoch 1/40
1402/1402 [==============================] - 278s 191ms/step - loss: 0.7203 - accuracy: 0.7845 - val_loss: 0.8395 - val_accuracy: 0.6621 - lr: 1.0000e-04
Epoch 2/40
1402/1402 [==============================] - 281s 192ms/step - loss: 0.5334 - accuracy: 0.8136 - val_loss: 0.6443 - val_accuracy: 0.7283 - lr: 1.0000e-04
Epoch 3/40
1402/1402 [==============================] - 281s 193ms/step - loss: 0.4981 - accuracy: 0.8254 - val_loss: 0.6099 - val_accuracy: 0.7506 - lr: 1.0000e-04
Epoch 4/40
1402/1402 [==============================] - 278s 191ms/step - loss: 0.4754 - accuracy: 0.8328 - val_loss: 0.5659 - val_accuracy: 0.7802 - lr: 1.0000e-04
Epoch 5/40
1402/1402 [==============================] - 278s 190ms/step - loss: 0.4686 - accuracy: 0.8358 - val_loss: 0.5520 - val_accuracy: 0.7812 - lr: 1.0000e-04
Epoch 6/40
1402/1402 [==============================] - 274s 188ms/step - loss: 0.4610 - accuracy: 0.8375 - val_loss: 0.5357 - val_accuracy: 0.7905 - lr: 1.0000e-04
Epoch 7/40

In [12]:
unfreeze_block(model_gap, "block5")

model_gap.compile(
    optimizer=Adam(FT_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_gap_fine_tune = model_gap.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FT_EPOCHS,
    callbacks=[early_stop, reduce_lr]
)

save_model_and_history(model_gap, history_gap_fine_tune, MODEL_PATH, "vgg16_gap_fine_tune")

All layers in block5 unfrozen.
Epoch 1/20
1402/1402 [==============================] - 264s 179ms/step - loss: 0.3896 - accuracy: 0.8692 - val_loss: 0.4744 - val_accuracy: 0.8321 - lr: 1.0000e-05
Epoch 2/20
1402/1402 [==============================] - 256s 176ms/step - loss: 0.3667 - accuracy: 0.8733 - val_loss: 0.4885 - val_accuracy: 0.8164 - lr: 1.0000e-05
Epoch 3/20
1402/1402 [==============================] - ETA: 0s - loss: 0.3480 - accuracy: 0.8824     
Epoch 3: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
1402/1402 [==============================] - 256s 175ms/step - loss: 0.3480 - accuracy: 0.8824 - val_loss: 0.4796 - val_accuracy: 0.8352 - lr: 1.0000e-05
Epoch 4/20
1402/1402 [==============================] - 256s 176ms/step - loss: 0.3143 - accuracy: 0.8921 - val_loss: 0.4635 - val_accuracy: 0.8373 - lr: 5.0000e-06
Epoch 5/20
1402/1402 [==============================] - 255s 175ms/step - loss: 0.2956 - accuracy: 0.8979 - val_loss: 0.4692 - val_accuracy: 

# Model 3

In [24]:
# Import class weights
with open(os.path.join(DATA_PATH, "class_weights.json"), "r") as f:
    class_weights = json.load(f)
    # Load as int keys (Required for training)
    class_weights = {int(k): v for k, v in class_weights.items()}
    
# Recreate model with same parameters
base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

for layer in base.layers:
    layer.trainable = False

# Using GAP layer
x = GlobalAveragePooling2D()(base.output)
x = Dense(512, activation="relu")(x)
# Add dropout to reduce overfitting
x = Dropout(DROPOUT_RATE)(x)
# Add final classification layer
output = Dense(3, activation="softmax")(x)
model_weighted = Model(inputs=base.input, outputs=output)

model_weighted.compile(
    optimizer=Adam(FE_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_weighted.summary()

Model: "model_8"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_10 (InputLayer)       [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0   

In [25]:
# Train classification layer and new layers
# Perform feature extraction
history_weighted_feature_extract = model_weighted.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FE_EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr]
)

save_model_and_history(model_weighted, history_weighted_feature_extract, MODEL_PATH, "vgg16_weighted_feature_extract")

Epoch 1/40
1402/1402 [==============================] - 220s 155ms/step - loss: 1.0834 - accuracy: 0.6960 - val_loss: 0.8723 - val_accuracy: 0.6458 - lr: 1.0000e-04
Epoch 2/40
1402/1402 [==============================] - 259s 177ms/step - loss: 0.7939 - accuracy: 0.7758 - val_loss: 0.7835 - val_accuracy: 0.6849 - lr: 1.0000e-04
Epoch 3/40
1402/1402 [==============================] - 254s 173ms/step - loss: 0.7433 - accuracy: 0.7965 - val_loss: 0.7503 - val_accuracy: 0.7004 - lr: 1.0000e-04
Epoch 4/40
1402/1402 [==============================] - 253s 173ms/step - loss: 0.7211 - accuracy: 0.8045 - val_loss: 0.7737 - val_accuracy: 0.7004 - lr: 1.0000e-04
Epoch 5/40
1402/1402 [==============================] - 256s 176ms/step - loss: 0.7043 - accuracy: 0.8100 - val_loss: 0.6998 - val_accuracy: 0.7192 - lr: 1.0000e-04
Epoch 6/40
1402/1402 [==============================] - 256s 175ms/step - loss: 0.6917 - accuracy: 0.8160 - val_loss: 0.6993 - val_accuracy: 0.7359 - lr: 1.0000e-04
Epoch 7/40

In [26]:
unfreeze_block(model_weighted, "block5")

model_weighted.compile(
    optimizer=Adam(FT_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_weighted_fine_tune = model_weighted.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FT_EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr]
)

save_model_and_history(model_weighted, history_weighted_fine_tune, MODEL_PATH, "vgg16_weighted_fine_tune")

All layers in block5 unfrozen.
Epoch 1/20
1402/1402 [==============================] - 252s 170ms/step - loss: 0.6052 - accuracy: 0.8394 - val_loss: 0.6301 - val_accuracy: 0.7585 - lr: 1.0000e-05
Epoch 2/20
1402/1402 [==============================] - 248s 170ms/step - loss: 0.5852 - accuracy: 0.8446 - val_loss: 0.5843 - val_accuracy: 0.7835 - lr: 1.0000e-05
Epoch 3/20
1402/1402 [==============================] - 249s 170ms/step - loss: 0.5567 - accuracy: 0.8494 - val_loss: 0.6758 - val_accuracy: 0.7492 - lr: 1.0000e-05
Epoch 4/20
1401/1402 [============================>.] - ETA: 0s - loss: 0.5259 - accuracy: 0.8614     
Epoch 4: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
1402/1402 [==============================] - 245s 168ms/step - loss: 0.5259 - accuracy: 0.8613 - val_loss: 0.6601 - val_accuracy: 0.7475 - lr: 1.0000e-05
Epoch 5/20
1402/1402 [==============================] - 244s 167ms/step - loss: 0.4729 - accuracy: 0.8761 - val_loss: 0.5307 - val_accuracy: 

# Model 4

In [15]:
# Recreate model with same parameters
base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

for layer in base.layers:
    layer.trainable = False
    
# SE block
x = create_se_block(base.output)

# Classification head
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(DROPOUT_RATE)(x)
output = Dense(3, activation="softmax")(x)

attention_model = Model(inputs=base.input, outputs=output)

attention_model.compile(
    optimizer=Adam(FE_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

attention_model.summary()

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_5 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 block1_conv1 (Conv2D)          (None, 224, 224, 64  1792        ['input_5[0][0]']                
                                )                                                                 
                                                                                                  
 block1_conv2 (Conv2D)          (None, 224, 224, 64  36928       ['block1_conv1[0][0]']           
                                )                                                           

In [13]:
# Train classification layer and new layers
# Perform feature extraction
history_attention_feature_extract = attention_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FE_EPOCHS,
    callbacks=[early_stop, reduce_lr]
)

save_model_and_history(attention_model, history_attention_feature_extract, MODEL_PATH, "vgg16_attention_feature_extract")

Epoch 1/40
1402/1402 [==============================] - 267s 183ms/step - loss: 0.5766 - accuracy: 0.8022 - val_loss: 0.7260 - val_accuracy: 0.6820 - lr: 1.0000e-04
Epoch 2/40
1402/1402 [==============================] - 271s 186ms/step - loss: 0.5130 - accuracy: 0.8171 - val_loss: 0.6356 - val_accuracy: 0.7378 - lr: 1.0000e-04
Epoch 3/40
1402/1402 [==============================] - 263s 180ms/step - loss: 0.4839 - accuracy: 0.8276 - val_loss: 0.5805 - val_accuracy: 0.7713 - lr: 1.0000e-04
Epoch 4/40
1402/1402 [==============================] - 262s 179ms/step - loss: 0.4698 - accuracy: 0.8326 - val_loss: 0.5659 - val_accuracy: 0.7847 - lr: 1.0000e-04
Epoch 5/40
1402/1402 [==============================] - 261s 179ms/step - loss: 0.4565 - accuracy: 0.8376 - val_loss: 0.5333 - val_accuracy: 0.8044 - lr: 1.0000e-04
Epoch 6/40
1402/1402 [==============================] - 260s 178ms/step - loss: 0.4505 - accuracy: 0.8405 - val_loss: 0.5304 - val_accuracy: 0.7934 - lr: 1.0000e-04
Epoch 7/40

In [11]:
unfreeze_block(attention_model, "block5")

attention_model.compile(
    optimizer=Adam(FT_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_attention_fine_tune = attention_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FT_EPOCHS,
    callbacks=[early_stop, reduce_lr]
)

save_model_and_history(attention_model, history_attention_fine_tune, MODEL_PATH, "vgg16_attention_fine_tune")

All layers in block5 unfrozen.
Epoch 1/20
1402/1402 [==============================] - 272s 180ms/step - loss: 0.3932 - accuracy: 0.8635 - val_loss: 0.4968 - val_accuracy: 0.8139 - lr: 1.0000e-05
Epoch 2/20
1402/1402 [==============================] - 255s 174ms/step - loss: 0.3705 - accuracy: 0.8722 - val_loss: 0.4682 - val_accuracy: 0.8364 - lr: 1.0000e-05
Epoch 3/20
1402/1402 [==============================] - 252s 172ms/step - loss: 0.3429 - accuracy: 0.8815 - val_loss: 0.4906 - val_accuracy: 0.8275 - lr: 1.0000e-05
Epoch 4/20
1402/1402 [==============================] - ETA: 0s - loss: 0.3235 - accuracy: 0.8892     
Epoch 4: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
1402/1402 [==============================] - 253s 173ms/step - loss: 0.3235 - accuracy: 0.8892 - val_loss: 0.4792 - val_accuracy: 0.8342 - lr: 1.0000e-05
Epoch 5/20
1402/1402 [==============================] - 263s 180ms/step - loss: 0.2852 - accuracy: 0.9009 - val_loss: 0.4982 - val_accuracy: 

# Model 5

In [4]:
train_ds_weighted, _, _ = load_datasets(os.path.join(DATA_PATH, "splits"),
                                        batch_size=BATCH_SIZE, 
                                        use_autotune=USE_AUTOTUNE, 
                                        img_size=IMG_SIZE, 
                                        weighted=True)

In [5]:
# Recreate model with same parameters
base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

for layer in base.layers:
    layer.trainable = False
    
# SE block
x = create_se_block(base.output)

# Classification head
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(DROPOUT_RATE)(x)
output = Dense(3, activation="softmax")(x)

attention_sample_weight_model = Model(inputs=base.input, outputs=output)

attention_sample_weight_model.compile(
    optimizer=Adam(FE_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

attention_sample_weight_model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 block1_conv1 (Conv2D)          (None, 224, 224, 64  1792        ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 block1_conv2 (Conv2D)          (None, 224, 224, 64  36928       ['block1_conv1[0][0]']           
                                )                                                             

In [6]:
# Train classification layer and new layers
# Perform feature extraction
history_attention_sample_weight_feature_extract = attention_sample_weight_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FE_EPOCHS,
    callbacks=[early_stop, reduce_lr]
)

save_model_and_history(attention_sample_weight_model,
                       history_attention_sample_weight_feature_extract,
                       MODEL_PATH,
                       "vgg16_attention_sample_weight_feature_extract")

Epoch 1/40
1402/1402 [==============================] - 291s 202ms/step - loss: 0.5814 - accuracy: 0.8010 - val_loss: 0.7463 - val_accuracy: 0.6725 - lr: 1.0000e-04
Epoch 2/40
1402/1402 [==============================] - 295s 210ms/step - loss: 0.5098 - accuracy: 0.8198 - val_loss: 0.6477 - val_accuracy: 0.7237 - lr: 1.0000e-04
Epoch 3/40
1402/1402 [==============================] - 302s 215ms/step - loss: 0.4845 - accuracy: 0.8269 - val_loss: 0.6028 - val_accuracy: 0.7566 - lr: 1.0000e-04
Epoch 4/40
1402/1402 [==============================] - 304s 217ms/step - loss: 0.4705 - accuracy: 0.8336 - val_loss: 0.6037 - val_accuracy: 0.7599 - lr: 1.0000e-04
Epoch 5/40
1402/1402 [==============================] - 295s 210ms/step - loss: 0.4617 - accuracy: 0.8360 - val_loss: 0.5382 - val_accuracy: 0.8015 - lr: 1.0000e-04
Epoch 6/40
1402/1402 [==============================] - 294s 209ms/step - loss: 0.4492 - accuracy: 0.8406 - val_loss: 0.5270 - val_accuracy: 0.7998 - lr: 1.0000e-04
Epoch 7/40

In [7]:
unfreeze_block(attention_sample_weight_model, "block5")

attention_sample_weight_model.compile(
    optimizer=Adam(FT_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_attention_sample_weight_fine_tune = attention_sample_weight_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FT_EPOCHS,
    callbacks=[early_stop, reduce_lr]
)

save_model_and_history(attention_sample_weight_model, 
                       history_attention_sample_weight_fine_tune, 
                       MODEL_PATH, 
                       "vgg16_attention_sample_weight_fine_tune")

All layers in block5 unfrozen.
Epoch 1/20
1402/1402 [==============================] - 303s 213ms/step - loss: 0.3984 - accuracy: 0.8635 - val_loss: 0.5374 - val_accuracy: 0.7959 - lr: 1.0000e-05
Epoch 2/20
1402/1402 [==============================] - 287s 205ms/step - loss: 0.3731 - accuracy: 0.8696 - val_loss: 0.5056 - val_accuracy: 0.8158 - lr: 1.0000e-05
Epoch 3/20
1402/1402 [==============================] - 311s 222ms/step - loss: 0.3487 - accuracy: 0.8809 - val_loss: 0.5035 - val_accuracy: 0.8236 - lr: 1.0000e-05
Epoch 4/20
1402/1402 [==============================] - 474s 338ms/step - loss: 0.3280 - accuracy: 0.8876 - val_loss: 0.4793 - val_accuracy: 0.8321 - lr: 1.0000e-05
Epoch 5/20
1402/1402 [==============================] - 341s 243ms/step - loss: 0.3061 - accuracy: 0.8950 - val_loss: 0.5097 - val_accuracy: 0.8296 - lr: 1.0000e-05
Epoch 6/20
1402/1402 [==============================] - ETA: 0s - loss: 0.2894 - accuracy: 0.9003   
Epoch 6: ReduceLROnPlateau reducing learnin

## Targeted Training

After evaluation, the UK and US datasets showed lower performance.  
To improve model performance on these subgroups, these models will be trained specifically on samples from the UK and US populations.  

In [5]:
# Load datasets
train_df, val_df, test_df = load_splits(os.path.join(DATA_PATH, "splits"))

# Load selected models
top_models = {
    "vgg16_gap_fe_targeted_train": load_model_and_history(MODEL_PATH, "vgg16_gap_feature_extract")[0],
    "vgg16_gap_ft_targeted_train": load_model_and_history(MODEL_PATH, "vgg16_gap_fine_tune")[0],
    "vgg16_ft_targeted_train": load_model_and_history(MODEL_PATH, "vgg16_fine_tune")[0]
}

In [6]:
# Define the grouping mapping
ethnicity_group_map = {
    "chinese": "Asian",
    "vietnamese": "Asian",
    "uk": "European",
    "portuguese": "European",
    "us": "US",
    "saudi": "Middle Eastern"
}

# Apply the mapping
train_df["ethnicity_grouped"] = train_df["ethnicity"].map(ethnicity_group_map)
val_df["ethnicity_grouped"] = val_df["ethnicity"].map(ethnicity_group_map)

# Filter underperforming groups
target_groups = ["European", "Middle Eastern"]

train_df_subset = train_df[train_df["ethnicity_grouped"].isin(target_groups)]
val_df_subset = val_df[val_df["ethnicity_grouped"].isin(target_groups)]
train_ds_subset = create_ds_from_df(train_df_subset, shuffle=True, augment=True)
val_ds_subset = create_ds_from_df(val_df_subset)

In [7]:
for model_name, model in top_models.items():
    print(f"Fine-tuning {model_name} on European + Middle Eastern subset")

    # Unfreeze top convolutional block
    unfreeze_block(model, "block5")

    # Compile model with fine-tuning learning rate
    model.compile(
        optimizer=Adam(FT_LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # Fit model
    history = model.fit(
        train_ds_subset,
        validation_data=val_ds_subset,
        epochs=FT_EPOCHS,
        callbacks=[early_stop, reduce_lr]
    )

    # Save fine-tuned model
    save_model_and_history(model, history, MODEL_PATH, model_name)
    print(f"Fine-tuning {model_name} completed")

Fine-tuning vgg16_gap_fe_targeted_train on European + Middle Eastern subset
All layers in block5 unfrozen.
Epoch 1/20
137/137 [==============================] - 73s 310ms/step - loss: 0.4577 - accuracy: 0.8269 - val_loss: 0.7340 - val_accuracy: 0.6976 - lr: 1.0000e-05
Epoch 2/20
137/137 [==============================] - 25s 179ms/step - loss: 0.3964 - accuracy: 0.8457 - val_loss: 0.6416 - val_accuracy: 0.7603 - lr: 1.0000e-05
Epoch 3/20
137/137 [==============================] - 8s 61ms/step - loss: 0.3471 - accuracy: 0.8659 - val_loss: 0.6632 - val_accuracy: 0.7581 - lr: 1.0000e-05
Epoch 4/20
136/137 [============================>.] - ETA: 0s - loss: 0.3229 - accuracy: 0.8851 
Epoch 4: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
137/137 [==============================] - 8s 59ms/step - loss: 0.3230 - accuracy: 0.8852 - val_loss: 0.6975 - val_accuracy: 0.7430 - lr: 1.0000e-05
Epoch 5/20
137/137 [==============================] - 8s 60ms/step - loss: 0.2726 - acc